# Colab All Models — Fair Benchmark & Extensions

Aligned to Agent 1's multi-asset benchmark contract. Core benchmark runs a single shared task (same target, dates, lookback, features, missing-data policy, and train-only scaling). Volatility and advanced PINN extensions write separate leaderboards, and ablations emit a summary table for reproducibility.

In [ ]:
#@title 1. Environment Setup
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/content/Dissertaion-Project')).resolve()

if 'google.colab' in sys.modules and not PROJECT_ROOT.exists():
    REPO_URL = 'https://github.com/must1f/Dissertaion-Project'
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'backend' / 'requirements.txt')], check=False)


In [ ]:
#@title 2. Imports & Run Directories
import json
from datetime import datetime

from scripts.train_models import (
    prepare_data,
    train_single_model,
    _write_track_leaderboards,
    _write_ablation_summary,
)
from src.data.pipeline import build_benchmark_dataset
from src.utils.config import get_config, get_research_config

RUN_ROOT = Path('results/colab_all_models')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUN_ROOT / datetime.utcnow().strftime('%Y%m%d_%H%M%S')
RUN_DIR.mkdir(parents=True, exist_ok=True)

core_models = ['lstm', 'gru', 'bilstm', 'attention_lstm', 'transformer', 'baseline_pinn', 'gbm', 'global']
volatility_models = ['vol_lstm', 'vol_gru', 'vol_transformer', 'vol_pinn', 'heston_pinn', 'stacked_vol_pinn']
advanced_pinn_models = ['stacked', 'residual', 'spectral_pinn', 'financial_pinn', 'financial_dp_pinn', 'financial_dual_phase_pinn', 'adaptive_dual_phase_pinn']
enable_volatility = False  # set True when a volatility target bundle is prepared
enable_advanced = True


In [ ]:
#@title 3. Build Shared Benchmark Bundles (core + optional volatility)
(
    train_ds,
    val_ds,
    test_ds,
    input_dim,
    scalers,
    feature_cols,
    dataset_meta,
    fairness_contract,
    regime_context,
) = prepare_data(research_mode=True, force_refresh=False)

# Persist core contract + metadata for reproducibility
with open(RUN_DIR / 'fairness_contract.json', 'w') as f:
    json.dump(fairness_contract, f, indent=2)
with open(RUN_DIR / 'dataset_metadata.json', 'w') as f:
    json.dump(dataset_meta, f, indent=2, default=str)

volatility_bundle = None
if enable_volatility:
    volatility_bundle = prepare_data(
        research_mode=True,
        force_refresh=False,
        target_override='realized_vol',
        target_column_override='realized_vol',
    )
    (
        vol_train_ds,
        vol_val_ds,
        vol_test_ds,
        vol_input_dim,
        vol_scalers,
        vol_feature_cols,
        vol_dataset_meta,
        vol_fairness_contract,
        vol_regime_context,
    ) = volatility_bundle
    with open(RUN_DIR / 'fairness_contract_volatility.json', 'w') as f:
        json.dump(vol_fairness_contract, f, indent=2)
    with open(RUN_DIR / 'dataset_metadata_volatility.json', 'w') as f:
        json.dump(vol_dataset_meta, f, indent=2, default=str)

run_metadata = {
    'run_dir': str(RUN_DIR),
    'timestamp_utc': datetime.utcnow().isoformat(),
    'models': {
        'core': core_models,
        'volatility': volatility_models if enable_volatility else [],
        'advanced': advanced_pinn_models if enable_advanced else [],
    },
    'fairness_contract_path': str(RUN_DIR / 'fairness_contract.json'),
    'volatility_contract_path': str(RUN_DIR / 'fairness_contract_volatility.json') if enable_volatility else None,
    'data_fingerprint': dataset_meta.get('fingerprint'),
    'volatility_fingerprint': vol_dataset_meta.get('fingerprint') if enable_volatility else None,
    'lookback': dataset_meta.get('lookback'),
    'horizon': dataset_meta.get('horizon'),
}
with open(RUN_DIR / 'run_metadata.json', 'w') as f:
    json.dump(run_metadata, f, indent=2)


In [ ]:
#@title 4. Core Benchmark Training (single shared task)
core_results = {}
for model in core_models:
    print(f'== Training {model} ==')
    core_results[model] = train_single_model(
        model_type=model,
        train_dataset=train_ds,
        val_dataset=val_ds,
        test_dataset=test_ds,
        input_dim=input_dim,
        research_mode=True,
        scalers=scalers,
        feature_cols=feature_cols,
        target_type=dataset_meta.get('target_type', 'next_day_log_return'),
        fairness_contract=fairness_contract,
        dataset_meta=dataset_meta,
        run_dir=RUN_DIR,
        regime_context=regime_context,
    )

leaderboards = _write_track_leaderboards(core_results, RUN_DIR)
leaderboards


In [ ]:
#@title 5. Extensions (separate tables; optional)
extension_results = dict(core_results)

if enable_advanced:
    for model in advanced_pinn_models:
        print(f'== Training {model} (advanced PINN) ==')
        extension_results[model] = train_single_model(
            model_type=model,
            train_dataset=train_ds,
            val_dataset=val_ds,
            test_dataset=test_ds,
            input_dim=input_dim,
            research_mode=True,
            scalers=scalers,
            feature_cols=feature_cols,
            target_type=dataset_meta.get('target_type', 'next_day_log_return'),
            fairness_contract=fairness_contract,
            dataset_meta=dataset_meta,
            run_dir=RUN_DIR,
            regime_context=regime_context,
        )

if enable_volatility:
    print('Volatility track enabled: using realized volatility bundle.')
    for model in volatility_models:
        extension_results[model] = train_single_model(
            model_type=model,
            train_dataset=vol_train_ds,
            val_dataset=vol_val_ds,
            test_dataset=vol_test_ds,
            input_dim=vol_input_dim,
            research_mode=True,
            scalers=vol_scalers,
            feature_cols=vol_feature_cols,
            target_type=vol_dataset_meta.get('target_type', 'realized_vol'),
            fairness_contract=vol_fairness_contract,
            dataset_meta=vol_dataset_meta,
            run_dir=RUN_DIR,
            regime_context=vol_regime_context,
        )
else:
    print('Volatility extension skipped (set enable_volatility=True to run with a volatility target dataset).')


In [ ]:
#@title 6. Leaderboards + Ablation Summary
all_leaderboards = _write_track_leaderboards(extension_results, RUN_DIR)
ablation_summary = _write_ablation_summary(RUN_DIR)
print('Leaderboards written:', all_leaderboards)
print('Ablation summary:', ablation_summary)
